In [ ]:
# 실습 준비 — 12주차 회귀분석
# 이 셀을 먼저 한 번 실행하세요. 데이터가 없으면 아래 셀들이 전부 실패합니다.
import os, pathlib, urllib.request

BASE = "https://raw.githubusercontent.com/aprilslab/statistics-lab/main/data/"
FILES = ["ch12_scores_reg.csv"]

pathlib.Path("data").mkdir(exist_ok=True)
for name in FILES:
    for dest in (pathlib.Path(name), pathlib.Path("data") / name):
        if not dest.exists():
            urllib.request.urlretrieve(BASE + name, dest)

# '../data/x.csv' 로 읽는 노트북 대응 — 상위 폴더에도 같은 data/ 를 걸어둔다.
# 절대경로(/data)로 박으면 cwd 가 /content 가 아닐 때 깨지므로 상대경로로 건다.
try:
    parent = pathlib.Path("..") / "data"
    if not parent.exists():
        os.symlink(pathlib.Path("data").resolve(), parent)
except OSError:
    pass

print("준비 완료:", ", ".join(FILES) if FILES else "(내려받을 데이터 없음)")


In [ ]:
# ── 참고용 셀입니다. 실행하지 않아도 됩니다. ─────────────────────
# 원래 이 셀은 Google Drive 를 연결해 거기 올려둔 데이터를 읽었습니다.
# 이 노트북은 맨 위 '실습 준비' 셀이 데이터를 직접 받아오므로 필요 없습니다.
# 그대로 실행하면 데이터가 없는 폴더로 옮겨 가 아래 셀이 전부 실패합니다.
# 나중에 내 Drive 의 데이터로 작업할 때를 위해 코드를 주석으로 남겨 둡니다.
#
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ── 참고용 셀입니다. 실행하지 않아도 됩니다. ─────────────────────
# 원래 이 셀은 Google Drive 를 연결해 거기 올려둔 데이터를 읽었습니다.
# 이 노트북은 맨 위 '실습 준비' 셀이 데이터를 직접 받아오므로 필요 없습니다.
# 그대로 실행하면 데이터가 없는 폴더로 옮겨 가 아래 셀이 전부 실패합니다.
# 나중에 내 Drive 의 데이터로 작업할 때를 위해 코드를 주석으로 남겨 둡니다.
#
# import os
# # work_dir = "/content/drive/MyDrive/데이터통계분석/source/data"
# work_dir = "{나의 data 경로}"
# os.chdir(work_dir)
# !ls

# 회귀분석

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

%precision 3
%matplotlib inline

In [ ]:
df = pd.read_csv('ch12_scores_reg.csv')
n = len(df)
print(n)
df.head()

## 단순회귀모형

In [ ]:
x = np.array(df['quiz'])
y = np.array(df['final_test'])
p = 1

In [ ]:
poly_fit = np.polyfit(x, y, 1)
poly_1d = np.poly1d(poly_fit)
xs = np.linspace(x.min(), x.max())
ys = poly_1d(xs)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)
ax.set_xlabel('quiz')
ax.set_ylabel('final test')
ax.plot(xs, ys, color='gray',
        label=f'{poly_fit[1]:.2f}+{poly_fit[0]:.2f}x')
ax.scatter(x, y)
ax.legend()

plt.show()

### 회귀분석에서의 가설

### statsmodels에 의한 회귀분석

In [ ]:
formula = 'final_test ~ quiz'
result = smf.ols(formula, df).fit()
result.summary()

### 회귀계수

In [ ]:
X = np.array([np.ones_like(x), x]).T
X

In [ ]:
beta0_hat, beta1_hat = np.linalg.lstsq(X, y)[0]
beta0_hat, beta1_hat

In [ ]:
y_hat = beta0_hat + beta1_hat * x
eps_hat = y - y_hat
eps_hat

In [ ]:
s_var = np.var(eps_hat, ddof=p+1)
s_var

In [ ]:
C0, C1 = np.diag(np.linalg.pinv(np.dot(X.T, X)))

In [ ]:
np.sqrt(s_var * C0), np.sqrt(s_var * C1)

In [ ]:
rv = stats.t(n-2)

lcl = beta0_hat - rv.isf(0.025) * np.sqrt(s_var * C0)
hcl = beta0_hat - rv.isf(0.975) * np.sqrt(s_var * C0)
lcl, hcl

In [ ]:
rv = stats.t(n-2)

lcl = beta1_hat - rv.isf(0.025) * np.sqrt(s_var * C1)
hcl = beta1_hat - rv.isf(0.975) * np.sqrt(s_var * C1)
lcl, hcl

In [ ]:
t = beta1_hat / np.sqrt(s_var * C1)
t

In [ ]:
(1 - rv.cdf(t)) * 2

In [ ]:
t = beta0_hat / np.sqrt(s_var * C0)
t

In [ ]:
(1 - rv.cdf(t)) * 2

## 중회귀모형

In [ ]:
formula = 'final_test ~ quiz + sleep_time'
result = smf.ols(formula, df).fit()
result.summary()

### 회귀계수

In [ ]:
x1 = df['quiz']
x2 = df['sleep_time']
y = df['final_test']
p = 2

In [ ]:
X = np.array([np.ones_like(x1), x1, x2]).T
beta0_hat, beta1_hat, beta2_hat = np.linalg.lstsq(X, y)[0]
beta0_hat, beta1_hat, beta2_hat

In [ ]:
y_hat = beta0_hat + beta1_hat * x1 + beta2_hat * x2
eps_hat = y - y_hat

In [ ]:
s_var = np.sum(eps_hat ** 2) / (n - p - 1)
C0, C1, C2 = np.diag(np.linalg.pinv(np.dot(X.T, X)))

In [ ]:
rv = stats.t(n-p-1)

lcl = beta2_hat - rv.isf(0.025) * np.sqrt(s_var * C2)
hcl = beta2_hat - rv.isf(0.975) * np.sqrt(s_var * C2)
lcl, hcl

### 가변수

In [ ]:
formula = 'final_test ~ quiz + sleep_time + school_method'
result = smf.ols(formula, df).fit()
result.summary()

## 모형의 선택

In [ ]:
x = np.array(df['quiz'])
y = np.array(df['final_test'])
p = 1

formula = 'final_test ~ quiz'
result = smf.ols(formula, df).fit()
result.summary()

In [ ]:
y_hat = np.array(result.fittedvalues)
y_hat

In [ ]:
eps_hat = np.array(result.resid)
eps_hat

In [ ]:
np.sum(eps_hat ** 2)

### 결정계수

In [ ]:
total_var = np.sum((y - np.mean(y))**2)
exp_var = np.sum((y_hat - np.mean(y))**2)
unexp_var = np.sum(eps_hat ** 2)

In [ ]:
total_var, exp_var + unexp_var

In [ ]:
exp_var / total_var

In [ ]:
np.corrcoef(x, y)[0, 1] ** 2

### 조정결정계수

In [ ]:
1 - (unexp_var / (n - p - 1)) / (total_var / (n - 1))

### F검정

In [ ]:
f = (exp_var / p)  / (unexp_var / (n - p - 1))
f

In [ ]:
rv = stats.f(p, n-p-1)
1 - rv.cdf(f)

### 최대 로그 우도와 AIC

In [ ]:
prob = 0.3
coin_result = [0, 1, 0, 0, 1]

rv = stats.bernoulli(prob)
L = np.prod(rv.pmf(coin_result))
L

In [ ]:
ps = np.linspace(0, 1, 100)
Ls = [np.prod(stats.bernoulli(prob).pmf(coin_result))
      for prob in ps]

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)
ax.plot(ps, Ls, label='likelihood function', color='gray')
ax.legend(fontsize=16)
plt.show()

In [ ]:
prob = 0.4
rv = stats.bernoulli(prob)
mll = np.sum(np.log(rv.pmf([0, 1, 0, 0, 1])))
mll

In [ ]:
rv = stats.norm(y_hat, np.sqrt(unexp_var / n))
mll = np.sum(np.log(rv.pdf(y)))
mll

In [ ]:
aic = -2 * mll + 2 * (p+1)
aic

In [ ]:
bic = -2 * mll + np.log(n) * (p+1)
bic

## 모형의 타당성

In [ ]:
formula = 'final_test ~ quiz + sleep_time'
result = smf.ols(formula, df).fit()
result.summary()

In [ ]:
eps_hat = np.array(result.resid)

### 정규성의 검정

In [ ]:
stats.skew(eps_hat)

In [ ]:
stats.kurtosis(eps_hat, fisher=False)

### 더빈-왓슨비

In [ ]:
np.sum(np.diff(eps_hat, 1) ** 2) / np.sum(eps_hat ** 2)

### 다중공선성

In [ ]:
df['mid_test'] = df['quiz'] * 2
df.head()

In [ ]:
formula = 'final_test ~ quiz + mid_test'
result = smf.ols(formula, df).fit()
result.summary()